In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.colors import ListedColormap, BoundaryNorm
from shapely.geometry import Point
from scipy.stats import gamma,norm,fisk,wilcoxon
from sklearn.cluster import KMeans
import re
import sys
from pathlib import Path
import logging
from glob import glob

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.append(str(project_root))

In [14]:
# Configuration
config = {
    'input_data_bow': project_root / 'data' / 'input_data' / 'for_report' / 'bow',
    'input_data_oldman': project_root / 'data' / 'input_data' / 'for_report' / 'oldman',
    'gap_filled_bow_canswe_data' : project_root / 'data' / 'output_data' / 'for_paper'/'FROSTBITE' /'bow_swe_gapfilled_data_final.nc',
    'gap_filled_oldman_canswe_data' : project_root / 'data' / 'output_data' / 'for_report'/'oldman' /'oldman_swe_gapfilled_data_final.nc',
    'shapefile': project_root / 'data' / 'input_data' / 'shapefiles'/'BowRiverBasin'/'Bow_elevation_combined.shp',
    'output_plots': project_root / 'data' / 'output_data' / 'for_report'}

# Data preparation

1. Create historic SWE dataset (Gap-filled CANSWE data for each basin).  

In [15]:
# open canswe data
bow_canswe_gapfilled_ds = xr.open_dataset(config['gap_filled_bow_canswe_data'])

oldman_canswe_gapfilled_ds = xr.open_dataset(config['gap_filled_oldman_canswe_data'])

# convert to dataframe and display
bow_canswe_gapfilled_df = bow_canswe_gapfilled_ds.to_dataframe()
bow_canswe_gapfilled_df.reset_index(inplace=True)
display(bow_canswe_gapfilled_df.head())

oldman_canswe_gapfilled_df = oldman_canswe_gapfilled_ds.to_dataframe()
oldman_canswe_gapfilled_df.reset_index(inplace=True)    
display(oldman_canswe_gapfilled_df.head())

,time,station_id,lat,lon,station_name,SWE
0,1980-01-01,ALE-05BA801,51.423084,-116.183693,BOW RIVER,NaN
1,1980-01-01,ALE-05BA802,51.437962,-116.181274,PIPESTONE UPPER,NaN
2,1980-01-01,ALE-05BA806,51.416618,-116.238274,MIRROR LAKE,NaN
3,1980-01-01,ALE-05BA808,51.424427,-116.213310,CHATEAU LAWN,NaN
4,1980-01-01,ALE-05BA810,51.474091,-116.102745,PTARMIGAN HUT,NaN


,time,station_id,lat,lon,station_name,SWE
0,1980-01-01,ALE-05AA801,49.279552,-114.372910,WEST CASTLE BUSH,NaN
1,1980-01-01,ALE-05AA803,49.745266,-114.612213,ALLISON PASS,NaN
2,1980-01-01,ALE-05AA805,49.266666,-114.349998,WEST CASTLE SNOW,NaN
3,1980-01-01,ALE-05AA806,49.816666,-114.633331,RACE HORSE CREEK,NaN
4,1980-01-01,ALE-05AA809,49.359940,-114.518234,GARDINER H.W.,NaN


2. Get latest data for the season from the downloaded data 

In [16]:
def extract_station_id(filepath):
    """
    Extract station id from filenames like:
    05BB803_SW_C.Corrected-Seasonal.csv
    """
    fname = Path(filepath).name
    m = re.search(r'([0-9A-Za-z]+)_', fname)
    return m.group(1) if m else fname


def find_data_start(filepath):
    """
    Find the row number right after the '#Timestamp;Value' line.
    """
    with open(filepath, 'r', encoding='utf-8-sig', errors='ignore') as f:
        for i, line in enumerate(f):
            if line.strip().startswith('#Timestamp;Value'):
                return i + 1
    raise ValueError(f"Could not find '#Timestamp;Value' header in {filepath}")


def read_alberta_timeseries(filepath, value_name='SWE'):
    """
    Read one Alberta seasonal CSV file.

    Returns columns:
    station_id, time, SWE
    """
    station_id = extract_station_id(filepath)
    skiprows = find_data_start(filepath)

    df = pd.read_csv(
        filepath,
        sep=';',
        skiprows=skiprows,
        names=['Timestamp', 'Value'],
        encoding='utf-8-sig'
    )

    df['time'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df[value_name] = pd.to_numeric(df['Value'], errors='coerce')

    df = df.dropna(subset=['time']).copy()
    df['station_id'] = station_id

    return df[['station_id', 'time', value_name]]


def add_season_year_nov_may(df, time_col='time'):
    """
    Keep only Nov-May data and assign season_year.

    Example:
    Nov-Dec 1984 -> season_year 1984
    Jan-May 1985 -> season_year 1984
    """
    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], errors='coerce')

    out = out.dropna(subset=[time_col]).copy()
    out['month'] = out[time_col].dt.month
    out['year'] = out[time_col].dt.year

    # Keep only Nov-May
    out = out[out['month'].isin([11, 12, 1, 2, 3, 4, 5])].copy()

    out['season_year'] = out['year']
    out.loc[out['month'].isin([1, 2, 3, 4, 5]), 'season_year'] = (
        out.loc[out['month'].isin([1, 2, 3, 4, 5]), 'year'] - 1
    )

    out = out.drop(columns=['month', 'year'])

    return out


def read_all_alberta_csvs(folderpath, value_name='SWE'):
    """
    Read all CSV files in a folder and combine them.

    Returns columns:
    station_id, time, SWE
    """
    folder = Path(folderpath)
    csv_files = sorted(folder.glob('*.csv'))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {folder}")

    all_dfs = []
    for file in csv_files:
        try:
            df = read_alberta_timeseries(file, value_name=value_name)
            all_dfs.append(df)
            print(f"Read: {file.name} -> {len(df)} rows")
        except Exception as e:
            print(f"Skipped {file.name}: {e}")

    if not all_dfs:
        raise ValueError("No valid CSV files were read.")

    combined = pd.concat(all_dfs, ignore_index=True)
    combined = combined.sort_values(['station_id', 'time']).reset_index(drop=True)

    return combined


def build_swe_table(folderpath):
    """
    Full pipeline:
    1. Read all CSVs in folder
    2. Add season_year
    3. Return final table with columns:
       station_id, time, SWE, season_year
    """
    df = read_all_alberta_csvs(folderpath, value_name='SWE')
    df = add_season_year_nov_may(df, time_col='time')

    df = df[['station_id', 'time', 'SWE', 'season_year']].copy()
    df = df.sort_values(['station_id', 'time']).reset_index(drop=True)

    return df

def daily_aggregate(df, value_col='SWE', how='last'):
    """
    Aggregate to daily data by station.
    For SWE, 'last' is usually the best choice.
    """
    if how == 'sum':
        out = (
            df.set_index('time')
              .groupby('station_id')[value_col]
              .resample('D')
              .sum()
              .reset_index()
        )
    elif how == 'mean':
        out = (
            df.set_index('time')
              .groupby('station_id')[value_col]
              .resample('D')
              .mean()
              .reset_index()
        )
    elif how == 'last':
        out = (
            df.set_index('time')
              .groupby('station_id')[value_col]
              .resample('D')
              .last()
              .reset_index()
        )
    else:
        raise ValueError("how must be 'sum', 'mean', or 'last'")


In [17]:
# Open CSVs, keep last SWE value per day, and create daily tables

# Bow basin
bow_raw = build_swe_table(config['input_data_bow'])
bow_swe_table = (
    bow_raw.sort_values(['station_id', 'season_year', 'time'])
           .set_index('time')
           .groupby(['station_id', 'season_year'])['SWE']
           .resample('D')
           .last()
           .reset_index()
)

# Oldman basin
oldman_raw = build_swe_table(config['input_data_oldman'])
oldman_swe_table = (
    oldman_raw.sort_values(['station_id', 'season_year', 'time'])
              .set_index('time')
              .groupby(['station_id', 'season_year'])['SWE']
              .resample('D')
              .last()
              .reset_index()
)

# keep only 2025 season year
bow_swe_table = bow_swe_table[bow_swe_table['season_year'] == 2025].copy()
oldman_swe_table = oldman_swe_table[oldman_swe_table['season_year'] == 2025].copy()

display(bow_raw.head())
display(oldman_raw.head())
display(bow_swe_table.head())
display(oldman_swe_table.head())
#display(daily_table.head())

Read: 05BB803_SW_C.Corrected-Seasonal.csv -> 355129 rows
Read: 05BF824_SW_C.Corrected-Seasonal.csv -> 339487 rows
Read: 05BJ805_SW_C.Corrected-Seasonal.csv -> 323828 rows
Read: 05BL811_SW_C.Corrected-Seasonal.csv -> 282719 rows
Read: 05BL812_SW_C.Corrected-Seasonal.csv -> 333428 rows
Read: 05CA805_SW_C.Corrected-Seasonal.csv -> 251112 rows
Read: 05AA809_SW_C.Corrected-Seasonal.csv -> 290591 rows
Read: 05AA817_SW_C.Corrected-Seasonal.csv -> 255260 rows
Read: 05AD803_SW_C.Corrected-Seasonal.csv -> 62356 rows


,station_id,time,SWE,season_year
0,05BB803,1984-03-12 10:00:00,472.0,1983
1,05BB803,1984-03-12 11:00:00,472.0,1983
2,05BB803,1984-03-12 12:00:00,472.0,1983
3,05BB803,1984-03-12 13:00:00,472.0,1983
4,05BB803,1984-03-12 14:00:00,472.0,1983


,station_id,time,SWE,season_year
0,05AA809,1984-04-01 00:00:00,427.0,1983
1,05AA809,1984-04-01 01:00:00,427.0,1983
2,05AA809,1984-04-01 02:00:00,428.0,1983
3,05AA809,1984-04-01 03:00:00,428.0,1983
4,05AA809,1984-04-01 04:00:00,428.0,1983


,station_id,season_year,time,SWE
8280,05BB803,2025,2025-11-01,87.0
8281,05BB803,2025,2025-11-02,95.0
8282,05BB803,2025,2025-11-03,92.0
8283,05BB803,2025,2025-11-04,91.0
8284,05BB803,2025,2025-11-05,95.0


,station_id,season_year,time,SWE
8070,05AA809,2025,2025-11-01,32.3
8071,05AA809,2025,2025-11-02,31.4
8072,05AA809,2025,2025-11-03,32.7
8073,05AA809,2025,2025-11-04,34.0
8074,05AA809,2025,2025-11-05,37.9


3. Extract monitoring stations historic data 

In [21]:
def extract_historic_by_station_ids(
    historic_df,
    current_df,
    historic_id_col="station_id",
    current_id_col="station_id",
):
    """
    Extract historic rows for stations present in current_df.
    Handles exact matches and prefixed historic IDs (e.g., ALE-05BB803 -> 05BB803).
    """
    target_ids = current_df[current_id_col].dropna().astype(str).unique().tolist()
    target_set = set(target_ids)

    historic_unique_ids = historic_df[historic_id_col].dropna().astype(str).unique().tolist()

    # Map historic station IDs to current station IDs
    id_map = {}
    for hid in historic_unique_ids:
        if hid in target_set:
            id_map[hid] = hid
            continue

        matches = [tid for tid in target_ids if hid.endswith(tid)]
        if len(matches) == 1:
            id_map[hid] = matches[0]

    # Filter and add matched station id
    out = historic_df[historic_df[historic_id_col].astype(str).isin(id_map.keys())].copy()
    out["source_station_id"] = out[historic_id_col].astype(str)
    out[current_id_col] = out["source_station_id"].map(id_map)

    out = out.sort_values([current_id_col, "time"]).reset_index(drop=True)
    return out



# Extract historic data corresponding to station IDs in latest-season tables
bow_historic_matched = extract_historic_by_station_ids(
    historic_df=bow_canswe_gapfilled_df,
    current_df=bow_swe_table,
)

# add season_year to historic matched data
bow_historic_matched = add_season_year_nov_may(bow_historic_matched, time_col='time')
#drop 1979 season year from historic matched data
bow_historic_matched = bow_historic_matched[bow_historic_matched['season_year'] != 1979].copy()

oldman_historic_matched = extract_historic_by_station_ids(
    historic_df=oldman_canswe_gapfilled_df,
    current_df=oldman_swe_table,
)
oldman_historic_matched = add_season_year_nov_may(oldman_historic_matched, time_col='time')
#drop 1979 season year from historic matched data
oldman_historic_matched = oldman_historic_matched[oldman_historic_matched['season_year'] != 1979].copy()


display(bow_historic_matched.head())
display(oldman_historic_matched.head())
print("Bow rows:", len(bow_historic_matched))
print("Oldman rows:", len(oldman_historic_matched))

,time,station_id,lat,lon,station_name,SWE,source_station_id,season_year
305,1980-11-01,05BB803,51.079578,-115.777596,SUNSHINE VILLAGE,NaN,ALE-05BB803,1980
306,1980-11-02,05BB803,51.079578,-115.777596,SUNSHINE VILLAGE,NaN,ALE-05BB803,1980
307,1980-11-03,05BB803,51.079578,-115.777596,SUNSHINE VILLAGE,NaN,ALE-05BB803,1980
308,1980-11-04,05BB803,51.079578,-115.777596,SUNSHINE VILLAGE,NaN,ALE-05BB803,1980
309,1980-11-05,05BB803,51.079578,-115.777596,SUNSHINE VILLAGE,NaN,ALE-05BB803,1980


,time,station_id,lat,lon,station_name,SWE,source_station_id,season_year
305,1980-11-01,05AA809,49.35994,-114.518234,GARDINER H.W.,NaN,ALE-05AA809,1980
306,1980-11-02,05AA809,49.35994,-114.518234,GARDINER H.W.,NaN,ALE-05AA809,1980
307,1980-11-03,05AA809,49.35994,-114.518234,GARDINER H.W.,NaN,ALE-05AA809,1980
308,1980-11-04,05AA809,49.35994,-114.518234,GARDINER H.W.,NaN,ALE-05AA809,1980
309,1980-11-05,05AA809,49.35994,-114.518234,GARDINER H.W.,NaN,ALE-05AA809,1980


Bow rows: 56034
Oldman rows: 28017


In [26]:
# check if nan in station_name_original
print("Unique Bow station_name_gapfilled values:")
print(bow_historic_matched['station_name'].unique())

print("Unique Oldman station_name_gapfilled values:")
print(oldman_historic_matched['station_name'].unique())


Unique Bow station_name_gapfilled values:
['SUNSHINE VILLAGE' 'THREE ISLE LAKE' 'LITTLE ELBOW' 'LOST CREEK'
 'MOUNT ODLUM III' 'SKOKI MOUNTAIN']
Unique Oldman station_name_gapfilled values:
['GARDINER H.W.' 'SOUTH RACE HORSE CREEK' 'AKAMINA']


4. Connect historic data with real-time data 

In [27]:
# add station_name column to bow_swe_table and oldman_swe_table by comparing station_id with historic matched data
bow_swe_table = bow_swe_table.merge(
    bow_historic_matched[['station_id', 'station_name']], 
    left_on='station_id', 
    right_on='station_id', 
    how='left'
)
oldman_swe_table = oldman_swe_table.merge(
    oldman_historic_matched[['station_id', 'station_name']],
    left_on='station_id',
    right_on='station_id',
    how='left'
)

# add season_2025_df data to the bottom of filtered_station
final_bow_df = pd.concat([bow_historic_matched, bow_swe_table], ignore_index=True)
final_oldman_df = pd.concat([oldman_historic_matched, oldman_swe_table], ignore_index=True)

# drop source_station_id,lat,lon column from final_bow_df and final_oldman_df
final_bow_df = final_bow_df.drop(columns=['source_station_id', 'lat', 'lon'])
final_oldman_df = final_oldman_df.drop(columns=['source_station_id', 'lat', 'lon'])


display(final_bow_df)
display(final_oldman_df)

MemoryError: Unable to allocate 659. GiB for an array with shape (88437957894,) and data type int64